In [ ]:
# Import Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm# Import Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

In [ ]:
# Check Device
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print("Using Device:", device)

In [ ]:
class VAE(nn.Module):

    def __init__(
        self,
        input_dim=784,
        hidden_dim=400,
        latent_dim=2
    ):
        super(VAE, self).__init__()

        # Encoder
        self.fc1 = nn.Linear(input_dim, hidden_dim)

        self.fc_mu = nn.Linear(
            hidden_dim,
            latent_dim
        )

        self.fc_logvar = nn.Linear(
            hidden_dim,
            latent_dim
        )

        # Decoder
        self.fc3 = nn.Linear(
            latent_dim,
            hidden_dim
        )

        self.fc4 = nn.Linear(
            hidden_dim,
            input_dim
        )

    def encode(self, x):

        h = F.relu(self.fc1(x))

        mu = self.fc_mu(h)

        logvar = self.fc_logvar(h)

        return mu, logvar

    def reparameterize(self, mu, logvar):

        std = torch.exp(0.5 * logvar)

        eps = torch.randn_like(std)

        return mu + eps * std

    def decode(self, z):

        h = F.relu(self.fc3(z))

        return torch.sigmoid(self.fc4(h))

    def forward(self, x):

        mu, logvar = self.encode(
            x.view(-1, 784)
        )

        z = self.reparameterize(
            mu,
            logvar
        )

        return self.decode(z), mu, logvar

In [ ]:
def vae_loss(recon_x, x, mu, logvar):

    # Reconstruction Loss
    BCE = F.binary_cross_entropy(
        recon_x,
        x.view(-1, 784),
        reduction='sum'
    )

    # KL Divergence
    KLD = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp()
    )

    return BCE + KLD

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

In [ ]:
batch_size = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [ ]:
latent_dim = 2

model = VAE(
    input_dim=784,
    hidden_dim=400,
    latent_dim=latent_dim
).to(device)

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [ ]:
def train_epoch(
    model,
    train_loader,
    optimizer,
    device
):

    model.train()

    train_loss = 0

    for batch_idx, (data, _) in enumerate(train_loader):

        data = data.to(device)

        optimizer.zero_grad()

        recon_batch, mu, logvar = model(data)

        loss = vae_loss(
            recon_batch,
            data,
            mu,
            logvar
        )

        loss.backward()

        train_loss += loss.item()

        optimizer.step()

    return train_loss / len(train_loader.dataset)

In [ ]:
def test_epoch(
    model,
    test_loader,
    device
):

    model.eval()

    test_loss = 0

    with torch.no_grad():

        for data, _ in test_loader:

            data = data.to(device)

            recon_batch, mu, logvar = model(data)

            test_loss += vae_loss(
                recon_batch,
                data,
                mu,
                logvar
            ).item()

    return test_loss / len(test_loader.dataset)

In [ ]:
epochs = 20

train_losses = []
test_losses = []

for epoch in range(1, epochs + 1):

    train_loss = train_epoch(
        model,
        train_loader,
        optimizer,
        device
    )

    test_loss = test_epoch(
        model,
        test_loader,
        device
    )

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    print(
        f"Epoch {epoch}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Test Loss: {test_loss:.4f}"
    )

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(test_losses, label='Test Loss')

plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.title('VAE Training Progress')

plt.legend()

plt.grid(True)

plt.show()

In [ ]:
model.eval()

with torch.no_grad():

    test_data, _ = next(iter(test_loader))

    test_data = test_data.to(device)

    recon_data, _, _ = model(test_data)

    n = 10

    plt.figure(figsize=(20, 4))

    for i in range(n):

        # Original
        ax = plt.subplot(2, n, i + 1)

        plt.imshow(
            test_data[i].cpu().squeeze(),
            cmap='gray'
        )

        plt.title("Original")

        plt.axis('off')

        # Reconstructed
        ax = plt.subplot(2, n, i + 1 + n)

        plt.imshow(
            recon_data[i].cpu().view(28, 28),
            cmap='gray'
        )

        plt.title("Reconstructed")

        plt.axis('off')

    plt.tight_layout()

    plt.show()

In [ ]:
model.eval()

latent_vectors = []
labels_list = []

with torch.no_grad():

    for data, labels in test_loader:

        data = data.to(device)

        mu, _ = model.encode(
            data.view(-1, 784)
        )

        latent_vectors.append(
            mu.cpu().numpy()
        )

        labels_list.append(
            labels.numpy()
        )

latent_vectors = np.concatenate(
    latent_vectors,
    axis=0
)

labels_list = np.concatenate(
    labels_list,
    axis=0
)

In [ ]:
plt.figure(figsize=(12, 10))

scatter = plt.scatter(
    latent_vectors[:, 0],
    latent_vectors[:, 1],
    c=labels_list,
    cmap='tab10',
    alpha=0.6,
    s=5
)

plt.colorbar(scatter, label='Digit')

plt.xlabel('Latent Dimension 1')
plt.ylabel('Latent Dimension 2')

plt.title('VAE Latent Space Visualization')

plt.grid(True)

plt.show()

In [ ]:
model.eval()

with torch.no_grad():

    n_samples = 20

    z = torch.randn(
        n_samples,
        latent_dim
    ).to(device)

    generated = model.decode(z)

    plt.figure(figsize=(20, 4))

    for i in range(n_samples):

        ax = plt.subplot(
            2,
            n_samples // 2,
            i + 1
        )

        plt.imshow(
            generated[i].cpu().view(28, 28),
            cmap='gray'
        )

        plt.title(f"Sample {i+1}")

        plt.axis('off')

    plt.tight_layout()

    plt.show()